<a href="https://colab.research.google.com/github/Nausheen1295/NorthStar-Coursework/blob/main/notebooks/Section_3_MongoDB/05_query_optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#installing pymongo
!pip install pymongo

In [ ]:
#importing library and loading data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

orders = pd.read_csv("/content/orders.csv")
deliveries = pd.read_csv("/content/deliveries.csv")
customers = pd.read_csv("/content/customers.csv")
drivers = pd.read_csv("/content/drivers.csv")
vehicles = pd.read_csv("/content/vehicles.csv")
hubs = pd.read_csv("/content/hubs.csv")
complaints = pd.read_csv("/content/complaints.csv")
incidents = pd.read_csv("/content/incidents.csv")
app_events = pd.read_csv("/content/app_events.csv")

In [ ]:
#checking the loaded data
import os
os.listdir("/content")

['.config',
 'deliveries.csv',
 'orders.csv',
 'hubs.csv',
 'customers.csv',
 'app_events.csv',
 'data_dictionary.csv',
 'complaints.csv',
 '.ipynb_checkpoints',
 'incidents.csv',
 'vehicles.csv',
 'README.txt',
 'drivers.csv',
 'sample_data']

In [ ]:
from getpass import getpass
from pymongo import MongoClient

In [ ]:
#connecting to mongodb atlas
connection_string = getpass("Paste MongoDB connection string: ")

client = MongoClient(connection_string)

db = client["northstar_db"]
collection = db["orders_operational_view"]

print(client.list_database_names())

Paste MongoDB connection string: ··········
['northstar_db', 'sample_analytics', 'sample_database', 'sample_flowerdata', 'sample_mflix', 'sample_supplies', 'admin', 'local']


In [ ]:
#collection.count_documents({})
collection.count_documents({})

1251

In [ ]:
#explain query before index
explain_before = collection.find(
    {"pickup_zone": "Central"}
).explain()

explain_before

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'pickup_zone': {'$eq': 'Central'}},
  'indexFilterSet': False,
  'queryHash': '2F27638F',
  'planCacheShapeHash': '2F27638F',
  'planCacheKey': 'D840907C',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'pickup_zone': 1},
    'indexName': 'pickup_zone_1',
    'isMultiKey': False,
    'multiKeyPaths': {'pickup_zone': []},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'pickup_zone': ['["Central", "Central"]']}}},
  'rejectedPlans': [{'isCached': False,
    'stage': 'FETCH',
    'inputStage': {'stage': 'IXSCAN',
     'keyPattern': {'picku

In [ ]:
# Create indexes for frequently searched fields

collection.create_index("order_id")
collection.create_index("service_type")
collection.create_index("pickup_zone")
collection.create_index("dropoff_zone")

collection.create_index("customer.customer_id")
collection.create_index("customer.customer_type")

collection.create_index("deliveries.delivery_status")
collection.create_index("deliveries.driver.driver_id")
collection.create_index("deliveries.vehicle.vehicle_id")
collection.create_index("deliveries.hub.hub_id")

collection.create_index("complaints.severity")
collection.create_index("complaints.status")

collection.create_index("app_events.event_type")

print("Indexes created successfully")

Indexes created successfully


In [ ]:
#list indexes
list(collection.list_indexes())

[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('order_id', 1)])), ('name', 'order_id_1')]),
 SON([('v', 2), ('key', SON([('customer.customer_id', 1)])), ('name', 'customer.customer_id_1')]),
 SON([('v', 2), ('key', SON([('service_type', 1)])), ('name', 'service_type_1')]),
 SON([('v', 2), ('key', SON([('pickup_zone', 1)])), ('name', 'pickup_zone_1')]),
 SON([('v', 2), ('key', SON([('dropoff_zone', 1)])), ('name', 'dropoff_zone_1')]),
 SON([('v', 2), ('key', SON([('deliveries.delivery_status', 1)])), ('name', 'deliveries.delivery_status_1')]),
 SON([('v', 2), ('key', SON([('deliveries.driver.driver_id', 1)])), ('name', 'deliveries.driver.driver_id_1')]),
 SON([('v', 2), ('key', SON([('deliveries.vehicle.vehicle_id', 1)])), ('name', 'deliveries.vehicle.vehicle_id_1')]),
 SON([('v', 2), ('key', SON([('deliveries.hub.hub_id', 1)])), ('name', 'deliveries.hub.hub_id_1')]),
 SON([('v', 2), ('key', SON([('complaints.severity', 1)])), ('name', 'com

In [ ]:
#explain query after index
explain_after = collection.find(
    {"pickup_zone": "Central"}
).explain()

explain_after

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'pickup_zone': {'$eq': 'Central'}},
  'indexFilterSet': False,
  'queryHash': '2F27638F',
  'planCacheShapeHash': '2F27638F',
  'planCacheKey': 'D840907C',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'pickup_zone': 1},
    'indexName': 'pickup_zone_1',
    'isMultiKey': False,
    'multiKeyPaths': {'pickup_zone': []},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'pickup_zone': ['["Central", "Central"]']}}},
  'rejectedPlans': [{'isCached': False,
    'stage': 'FETCH',
    'inputStage': {'stage': 'IXSCAN',
     'keyPattern': {'picku

In [ ]:
#filter by zone and service type
list(collection.find(
    {
        "pickup_zone": "Central",
        "service_type": "Business"
    },
    {
        "order_id": 1,
        "pickup_zone": 1,
        "dropoff_zone": 1,
        "service_type": 1
    }
).limit(5))

[{'_id': ObjectId('6a00406604f081013780edc0'),
  'order_id': 'O00064',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'RiverSide'},
 {'_id': ObjectId('6a00406604f081013780ee1e'),
  'order_id': 'O00158',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'West'},
 {'_id': ObjectId('6a00406604f081013780ee82'),
  'order_id': 'O00258',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'West'},
 {'_id': ObjectId('6a00406604f081013780eef0'),
  'order_id': 'O00368',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'north'},
 {'_id': ObjectId('6a00406604f081013780ef1b'),
  'order_id': 'O00411',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'Central'}]

In [ ]:
#failed deliveries
list(collection.find(
    {"deliveries.delivery_status": "Failed"},
    {
        "order_id": 1,
        "service_type": 1,
        "deliveries.delivery_status": 1,
        "deliveries.customer_rating_post_delivery": 1
    }
).limit(5))

[{'_id': ObjectId('6a00406604f081013780ed97'),
  'order_id': 'O00023',
  'service_type': 'Passenger',
  'deliveries': [{'delivery_status': 'Failed',
    'customer_rating_post_delivery': 2.38}]},
 {'_id': ObjectId('6a00406604f081013780ed9f'),
  'order_id': 'O00031',
  'service_type': 'Passenger',
  'deliveries': [{'delivery_status': 'Failed',
    'customer_rating_post_delivery': 1.41}]},
 {'_id': ObjectId('6a00406604f081013780eda1'),
  'order_id': 'O00033',
  'service_type': 'Retail',
  'deliveries': [{'delivery_status': 'Failed',
    'customer_rating_post_delivery': 3.2}]},
 {'_id': ObjectId('6a00406604f081013780eda3'),
  'order_id': 'O00035',
  'service_type': 'Business',
  'deliveries': [{'delivery_status': 'Failed',
    'customer_rating_post_delivery': 1.99}]},
 {'_id': ObjectId('6a00406604f081013780edc6'),
  'order_id': 'O00070',
  'service_type': 'Retail',
  'deliveries': [{'delivery_status': 'Failed',
    'customer_rating_post_delivery': 2.14}]}]

In [ ]:
#high-severity complaints
list(collection.find(
    {"complaints.severity": "High"},
    {
        "order_id": 1,
        "customer.customer_id": 1,
        "complaints": 1
    }
).limit(3))

[{'_id': ObjectId('6a00406604f081013780ed87'),
  'order_id': 'O00007',
  'customer': {'customer_id': 'C0001'},
  'complaints': [{'complaint_id': 'CP0096',
    'complaint_type': 'AppIssue',
    'channel': 'App',
    'severity': 'High',
    'created_at': '2024-05-12 21:32:00',
    'status': 'Resolved',
    'resolution_days': 22,
    'compensation_amount': 43.9}]},
 {'_id': ObjectId('6a00406604f081013780ed9f'),
  'order_id': 'O00031',
  'customer': {'customer_id': 'C0573'},
  'complaints': [{'complaint_id': 'CP0077',
    'complaint_type': 'Billing',
    'channel': 'Email',
    'severity': 'High',
    'created_at': '2024-11-02 03:17:00',
    'status': 'Resolved',
    'resolution_days': 13,
    'compensation_amount': 58.46},
   {'complaint_id': 'CP0279',
    'complaint_type': 'Billing',
    'channel': 'App',
    'severity': 'Medium',
    'created_at': '2024-10-27 03:17:00',
    'status': 'Resolved',
    'resolution_days': 1,
    'compensation_amount': 19.87}]},
 {'_id': ObjectId('6a00406604

In [ ]:
#driver investigation
list(collection.find(
    {"deliveries.driver.driver_id": "D004"},
    {
        "order_id": 1,
        "deliveries.driver": 1,
        "deliveries.delivery_status": 1,
        "deliveries.manual_route_override_count": 1
    }
).limit(5))

[{'_id': ObjectId('6a00406604f081013780edc3'),
  'order_id': 'O00067',
  'deliveries': [{'delivery_status': 'OnTime',
    'manual_route_override_count': 0,
    'driver': {'driver_id': 'D004',
     'base_zone': 'Airport',
     'employment_type': 'PartTime',
     'years_experience': 13,
     'training_score': 88.9,
     'driver_rating': 4.75,
     'shift_preference': 'Morning',
     'active_flag': 1}}]},
 {'_id': ObjectId('6a00406604f081013780ee8d'),
  'order_id': 'O00269',
  'deliveries': [{'delivery_status': 'Failed',
    'manual_route_override_count': 2,
    'driver': {'driver_id': 'D004',
     'base_zone': 'Airport',
     'employment_type': 'PartTime',
     'years_experience': 13,
     'training_score': 88.9,
     'driver_rating': 4.75,
     'shift_preference': 'Morning',
     'active_flag': 1}}]},
 {'_id': ObjectId('6a00406604f081013780efb2'),
  'order_id': 'O00562',
  'deliveries': [{'delivery_status': 'OnTime',
    'manual_route_override_count': 2,
    'driver': {'driver_id': 'D00

In [ ]:
# Check available values before writing filtered queries

print("Service types:", collection.distinct("service_type"))
print("Pickup zones:", collection.distinct("pickup_zone"))
print("Customer types:", collection.distinct("customer.customer_type"))
print("Delivery statuses:", collection.distinct("deliveries.delivery_status"))
print("Complaint severities:", collection.distinct("complaints.severity"))
print("Hub IDs:", collection.distinct("deliveries.hub.hub_id"))

Service types: ['Business', 'Medical', 'Parcel', 'Passenger', 'Retail']
Pickup zones: ['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'EAST', 'East', 'NORTH', 'North', 'RiverSide', 'Riverside', 'SOUTH', 'South', 'WEST', 'West', 'north']
Customer types: ['Consumer', 'Enterprise', 'SME']
Delivery statuses: [None, 'Delayed', 'Failed', 'OnTime']
Complaint severities: [None, 'High', 'Low', 'Medium']
Hub IDs: [None, 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08']


In [ ]:
# Explain plan for service type query
service_explain = collection.find(
    {"service_type": "Delivery"}
).explain()
service_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'service_type': {'$eq': 'Delivery'}},
  'indexFilterSet': False,
  'queryHash': '2CC52BBC',
  'planCacheShapeHash': '2CC52BBC',
  'planCacheKey': '66DB895E',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'service_type': 1},
    'indexName': 'service_type_1',
    'isMultiKey': False,
    'multiKeyPaths': {'service_type': []},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'service_type': ['["Delivery", "Delivery"]']}}},
  'rejectedPlans': []},
 'executionStats': {'executionSuccess': True,
  'nReturned': 0,
  'executionTimeMillis': 0,
  

In [ ]:
# Optimised query: search by service type
service_orders = list(collection.find(
    {"service_type": "Business"},
    {
        "_id": 0,
        "order_id": 1,
        "service_type": 1,
        "pickup_zone": 1,
        "dropoff_zone": 1,
        "order_value": 1
    }
).limit(5))
service_orders

[{'order_id': 'O00007',
  'service_type': 'Business',
  'pickup_zone': 'CENTRAL',
  'dropoff_zone': 'Airport',
  'order_value': 76.12},
 {'order_id': 'O00012',
  'service_type': 'Business',
  'pickup_zone': 'East',
  'dropoff_zone': 'Airport',
  'order_value': 135.67},
 {'order_id': 'O00015',
  'service_type': 'Business',
  'pickup_zone': 'SOUTH',
  'dropoff_zone': 'East',
  'order_value': 145.58},
 {'order_id': 'O00035',
  'service_type': 'Business',
  'pickup_zone': 'north',
  'dropoff_zone': 'CENTRAL',
  'order_value': 209.07},
 {'order_id': 'O00044',
  'service_type': 'Business',
  'pickup_zone': 'West',
  'dropoff_zone': 'Riverside',
  'order_value': 61.45}]

In [ ]:
# Optimised query: high-priority orders by service type
priority_orders = list(collection.find(
    {
        "service_type": "Business",
        "priority_level": "High"
    },
    {
        "_id": 0,
        "order_id": 1,
        "service_type": 1,
        "priority_level": 1,
        "pickup_zone": 1,
        "dropoff_zone": 1,
        "order_value": 1
    }
).limit(5))
priority_orders

[{'order_id': 'O00015',
  'service_type': 'Business',
  'pickup_zone': 'SOUTH',
  'dropoff_zone': 'East',
  'priority_level': 'High',
  'order_value': 145.58},
 {'order_id': 'O00064',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'RiverSide',
  'priority_level': 'High',
  'order_value': 45.43},
 {'order_id': 'O00113',
  'service_type': 'Business',
  'pickup_zone': 'EAST',
  'dropoff_zone': 'north',
  'priority_level': 'High',
  'order_value': 113.54},
 {'order_id': 'O00114',
  'service_type': 'Business',
  'pickup_zone': 'SOUTH',
  'dropoff_zone': 'Riverside',
  'priority_level': 'High',
  'order_value': 15.52},
 {'order_id': 'O00141',
  'service_type': 'Business',
  'pickup_zone': 'North',
  'dropoff_zone': 'SOUTH',
  'priority_level': 'High',
  'order_value': 83.09}]

In [ ]:
# Optimised query: zone and service type search
zone_service_orders = list(collection.find(
    {
        "pickup_zone": "Central",
        "service_type": "Business"
    },
    {
        "_id": 0,
        "order_id": 1,
        "pickup_zone": 1,
        "dropoff_zone": 1,
        "service_type": 1,
        "order_value": 1
    }
).limit(5))
zone_service_orders

[{'order_id': 'O00064',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'RiverSide',
  'order_value': 45.43},
 {'order_id': 'O00158',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'West',
  'order_value': 67.81},
 {'order_id': 'O00258',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'West',
  'order_value': 98.81},
 {'order_id': 'O00368',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'north',
  'order_value': 89.38},
 {'order_id': 'O00411',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'Central',
  'order_value': 71.11}]

In [ ]:
zone_service_explain = collection.find(
    {
        "pickup_zone": "Central",
        "service_type": "Business"
    }
).explain()
zone_service_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'$and': [{'pickup_zone': {'$eq': 'Central'}},
    {'service_type': {'$eq': 'Business'}}]},
  'indexFilterSet': False,
  'queryHash': 'CA66A2F2',
  'planCacheShapeHash': 'CA66A2F2',
  'planCacheKey': '4CEAEA12',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'pickup_zone': 1, 'service_type': 1},
    'indexName': 'pickup_zone_1_service_type_1',
    'isMultiKey': False,
    'multiKeyPaths': {'pickup_zone': [], 'service_type': []},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'pickup_zone': ['["Central", "Central"]'],
     'service_type':

In [ ]:
# Optimised query: customer type and service type
customer_service_orders = list(collection.find(
    {
        "customer.customer_type": "Consumer",
        "service_type": "Business"
    },
    {
        "_id": 0,
        "order_id": 1,
        "service_type": 1,
        "customer.customer_id": 1,
        "customer.customer_type": 1,
        "order_value": 1
    }
).limit(5))
customer_service_orders

[{'order_id': 'O00012',
  'service_type': 'Business',
  'order_value': 135.67,
  'customer': {'customer_id': 'C0567', 'customer_type': 'Consumer'}},
 {'order_id': 'O00015',
  'service_type': 'Business',
  'order_value': 145.58,
  'customer': {'customer_id': 'C0300', 'customer_type': 'Consumer'}},
 {'order_id': 'O00035',
  'service_type': 'Business',
  'order_value': 209.07,
  'customer': {'customer_id': 'C0524', 'customer_type': 'Consumer'}},
 {'order_id': 'O00044',
  'service_type': 'Business',
  'order_value': 61.45,
  'customer': {'customer_id': 'C0227', 'customer_type': 'Consumer'}},
 {'order_id': 'O00051',
  'service_type': 'Business',
  'order_value': 14.48,
  'customer': {'customer_id': 'C0185', 'customer_type': 'Consumer'}}]

In [ ]:
customer_service_explain = collection.find(
    {
        "customer.customer_type": "Consumer",
        "service_type": "Business"
    }
).explain()
customer_service_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'$and': [{'customer.customer_type': {'$eq': 'Consumer'}},
    {'service_type': {'$eq': 'Business'}}]},
  'indexFilterSet': False,
  'queryHash': '239FA474',
  'planCacheShapeHash': '239FA474',
  'planCacheKey': '84F75738',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'filter': {'customer.customer_type': {'$eq': 'Consumer'}},
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'service_type': 1},
    'indexName': 'service_type_1',
    'isMultiKey': False,
    'multiKeyPaths': {'service_type': []},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'service_type': ['["Business", "Business

In [ ]:
# Optimised query: failed deliveries
failed_delivery_cases = list(collection.find(
    {"deliveries.delivery_status": "Failed"},
    {
        "_id": 0,
        "order_id": 1,
        "service_type": 1,
        "pickup_zone": 1,
        "deliveries.delivery_id": 1,
        "deliveries.delivery_status": 1,
        "deliveries.customer_rating_post_delivery": 1
    }
).limit(5))
failed_delivery_cases

[{'order_id': 'O00023',
  'service_type': 'Passenger',
  'pickup_zone': 'north',
  'deliveries': [{'delivery_id': 'DL00831',
    'delivery_status': 'Failed',
    'customer_rating_post_delivery': 2.38}]},
 {'order_id': 'O00031',
  'service_type': 'Passenger',
  'pickup_zone': 'EAST',
  'deliveries': [{'delivery_id': 'DL00862',
    'delivery_status': 'Failed',
    'customer_rating_post_delivery': 1.41}]},
 {'order_id': 'O00033',
  'service_type': 'Retail',
  'pickup_zone': 'NORTH',
  'deliveries': [{'delivery_id': 'DL00792',
    'delivery_status': 'Failed',
    'customer_rating_post_delivery': 3.2}]},
 {'order_id': 'O00035',
  'service_type': 'Business',
  'pickup_zone': 'north',
  'deliveries': [{'delivery_id': 'DL00492',
    'delivery_status': 'Failed',
    'customer_rating_post_delivery': 1.99}]},
 {'order_id': 'O00070',
  'service_type': 'Retail',
  'pickup_zone': 'Ctr',
  'deliveries': [{'delivery_id': 'DL00378',
    'delivery_status': 'Failed',
    'customer_rating_post_delivery': 

In [ ]:
failed_delivery_explain = collection.find(
    {"deliveries.delivery_status": "Failed"}
).explain()

failed_delivery_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'deliveries.delivery_status': {'$eq': 'Failed'}},
  'indexFilterSet': False,
  'queryHash': '19C14B90',
  'planCacheShapeHash': '19C14B90',
  'planCacheKey': '6E45C474',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'deliveries.delivery_status': 1},
    'indexName': 'deliveries.delivery_status_1',
    'isMultiKey': True,
    'multiKeyPaths': {'deliveries.delivery_status': ['deliveries']},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'deliveries.delivery_status': ['["Failed", "Failed"]']}}},
  'rejectedPlans': []},
 'executionStats': 

In [ ]:
collection.distinct("deliveries.hub.hub_id")

[None, 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08']

In [ ]:
# Optimised query: orders linked to a specific hub
hub_orders = list(collection.find(
    {"deliveries.hub.hub_id": "H01"},
    {
        "_id": 0,
        "order_id": 1,
        "service_type": 1,
        "deliveries.delivery_id": 1,
        "deliveries.delivery_status": 1,
        "deliveries.hub.hub_id": 1,
        "deliveries.hub.hub_name": 1,
        "deliveries.hub.zone": 1
    }
).limit(5))
hub_orders

[{'order_id': 'O00001',
  'service_type': 'Passenger',
  'deliveries': [{'delivery_id': 'DL00937',
    'delivery_status': 'OnTime',
    'hub': {'hub_id': 'H01', 'hub_name': 'North Exchange', 'zone': 'North'}}]},
 {'order_id': 'O00009',
  'service_type': 'Retail',
  'deliveries': [{'delivery_id': 'DL00042',
    'delivery_status': 'OnTime',
    'hub': {'hub_id': 'H01', 'hub_name': 'North Exchange', 'zone': 'North'}}]},
 {'order_id': 'O00014',
  'service_type': 'Passenger',
  'deliveries': [{'delivery_id': 'DL00441',
    'delivery_status': 'OnTime',
    'hub': {'hub_id': 'H01', 'hub_name': 'North Exchange', 'zone': 'North'}}]},
 {'order_id': 'O00020',
  'service_type': 'Retail',
  'deliveries': [{'delivery_id': 'DL00752',
    'delivery_status': 'OnTime',
    'hub': {'hub_id': 'H01', 'hub_name': 'North Exchange', 'zone': 'North'}}]},
 {'order_id': 'O00022',
  'service_type': 'Retail',
  'deliveries': [{'delivery_id': 'DL00691',
    'delivery_status': 'Delayed',
    'hub': {'hub_id': 'H01',

In [ ]:
hub_explain = collection.find(
    {"deliveries.hub.hub_id": "H01"}
).explain()
hub_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'deliveries.hub.hub_id': {'$eq': 'H01'}},
  'indexFilterSet': False,
  'queryHash': 'FEA20880',
  'planCacheShapeHash': 'FEA20880',
  'planCacheKey': '1EE8F3BA',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'deliveries.hub.hub_id': 1},
    'indexName': 'deliveries.hub.hub_id_1',
    'isMultiKey': True,
    'multiKeyPaths': {'deliveries.hub.hub_id': ['deliveries']},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'deliveries.hub.hub_id': ['["H01", "H01"]']}}},
  'rejectedPlans': []},
 'executionStats': {'executionSuccess': True,
  'nRet

In [ ]:
collection.distinct("deliveries.driver.driver_id")

[None,
 'D001',
 'D002',
 'D003',
 'D004',
 'D005',
 'D006',
 'D007',
 'D008',
 'D009',
 'D010',
 'D011',
 'D012',
 'D013',
 'D014',
 'D015',
 'D016',
 'D017',
 'D018',
 'D019',
 'D020',
 'D021',
 'D022',
 'D023',
 'D024',
 'D025',
 'D026',
 'D027',
 'D028',
 'D029',
 'D030',
 'D031',
 'D032',
 'D033',
 'D034',
 'D035',
 'D036',
 'D037',
 'D038',
 'D039',
 'D040',
 'D041',
 'D042',
 'D043',
 'D044',
 'D045',
 'D046',
 'D047',
 'D048',
 'D049',
 'D050',
 'D051',
 'D052',
 'D053',
 'D054',
 'D055',
 'D056',
 'D057',
 'D058',
 'D059',
 'D060',
 'D061',
 'D062',
 'D063',
 'D064',
 'D065',
 'D066',
 'D067',
 'D068',
 'D069',
 'D070',
 'D071',
 'D072',
 'D073',
 'D074',
 'D075',
 'D076',
 'D077',
 'D078',
 'D079',
 'D080',
 'D081',
 'D082',
 'D083',
 'D084',
 'D085',
 'D086',
 'D087',
 'D088',
 'D089',
 'D090',
 'D091',
 'D092',
 'D093',
 'D094',
 'D095',
 'D096',
 'D097',
 'D098',
 'D099',
 'D100',
 'D101',
 'D102',
 'D103',
 'D104',
 'D105',
 'D106',
 'D107',
 'D108',
 'D109',
 'D110',
 'D

In [ ]:
# Optimised query: deliveries handled by a specific driver
driver_orders = list(collection.find(
    {"deliveries.driver.driver_id": "D004"},
    {
        "_id": 0,
        "order_id": 1,
        "service_type": 1,
        "deliveries.delivery_id": 1,
        "deliveries.delivery_status": 1,
        "deliveries.manual_route_override_count": 1,
        "deliveries.driver.driver_id": 1,
        "deliveries.driver.training_score": 1,
        "deliveries.driver.driver_rating": 1
    }
).limit(5))
driver_orders

[{'order_id': 'O00067',
  'service_type': 'Passenger',
  'deliveries': [{'delivery_id': 'DL00474',
    'delivery_status': 'OnTime',
    'manual_route_override_count': 0,
    'driver': {'driver_id': 'D004',
     'training_score': 88.9,
     'driver_rating': 4.75}}]},
 {'order_id': 'O00269',
  'service_type': 'Passenger',
  'deliveries': [{'delivery_id': 'DL00103',
    'delivery_status': 'Failed',
    'manual_route_override_count': 2,
    'driver': {'driver_id': 'D004',
     'training_score': 88.9,
     'driver_rating': 4.75}}]},
 {'order_id': 'O00562',
  'service_type': 'Passenger',
  'deliveries': [{'delivery_id': 'DL00470',
    'delivery_status': 'OnTime',
    'manual_route_override_count': 2,
    'driver': {'driver_id': 'D004',
     'training_score': 88.9,
     'driver_rating': 4.75}}]},
 {'order_id': 'O00631',
  'service_type': 'Business',
  'deliveries': [{'delivery_id': 'DL00098',
    'delivery_status': 'Failed',
    'manual_route_override_count': 1,
    'driver': {'driver_id': 'D

In [ ]:
driver_explain = collection.find(
    {"deliveries.driver.driver_id": "D004"}
).explain()
driver_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'deliveries.driver.driver_id': {'$eq': 'D004'}},
  'indexFilterSet': False,
  'queryHash': '19366BD0',
  'planCacheShapeHash': '19366BD0',
  'planCacheKey': 'A42CE2F7',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'deliveries.driver.driver_id': 1},
    'indexName': 'deliveries.driver.driver_id_1',
    'isMultiKey': True,
    'multiKeyPaths': {'deliveries.driver.driver_id': ['deliveries']},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'deliveries.driver.driver_id': ['["D004", "D004"]']}}},
  'rejectedPlans': []},
 'executionStats': {

In [ ]:
# Optimised query: high-severity complaint cases
high_complaint_cases = list(collection.find(
    {"complaints.severity": "High"},
    {
        "_id": 0,
        "order_id": 1,
        "customer.customer_id": 1,
        "customer.customer_type": 1,
        "complaints.complaint_id": 1,
        "complaints.complaint_type": 1,
        "complaints.severity": 1,
        "complaints.status": 1,
        "complaints.compensation_amount": 1
    }
).limit(5))
high_complaint_cases

[{'order_id': 'O00007',
  'customer': {'customer_id': 'C0001', 'customer_type': 'SME'},
  'complaints': [{'complaint_id': 'CP0096',
    'complaint_type': 'AppIssue',
    'severity': 'High',
    'status': 'Resolved',
    'compensation_amount': 43.9}]},
 {'order_id': 'O00031',
  'customer': {'customer_id': 'C0573', 'customer_type': 'SME'},
  'complaints': [{'complaint_id': 'CP0077',
    'complaint_type': 'Billing',
    'severity': 'High',
    'status': 'Resolved',
    'compensation_amount': 58.46},
   {'complaint_id': 'CP0279',
    'complaint_type': 'Billing',
    'severity': 'Medium',
    'status': 'Resolved',
    'compensation_amount': 19.87}]},
 {'order_id': 'O00048',
  'customer': {'customer_id': 'C0145', 'customer_type': 'Consumer'},
  'complaints': [{'complaint_id': 'CP0214',
    'complaint_type': 'MissedPickup',
    'severity': 'High',
    'status': 'Resolved',
    'compensation_amount': 61.85}]},
 {'order_id': 'O00050',
  'customer': {'customer_id': 'C0544', 'customer_type': 'Con

In [ ]:
complaint_explain = collection.find(
    {"complaints.severity": "High"}
).explain()
complaint_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'complaints.severity': {'$eq': 'High'}},
  'indexFilterSet': False,
  'queryHash': 'F92575E3',
  'planCacheShapeHash': 'F92575E3',
  'planCacheKey': '539EEB7D',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'complaints.severity': 1},
    'indexName': 'complaints.severity_1',
    'isMultiKey': True,
    'multiKeyPaths': {'complaints.severity': ['complaints']},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'complaints.severity': ['["High", "High"]']}}},
  'rejectedPlans': []},
 'executionStats': {'executionSuccess': True,
  'nReturned':

In [ ]:
# Optimised query: open complaints
open_complaint_cases = list(collection.find(
    {"complaints.status": "Open"},
    {
        "_id": 0,
        "order_id": 1,
        "customer.customer_id": 1,
        "complaints.complaint_id": 1,
        "complaints.complaint_type": 1,
        "complaints.status": 1,
        "complaints.severity": 1
    }
).limit(5))
open_complaint_cases

[{'order_id': 'O00003',
  'customer': {'customer_id': 'C0161'},
  'complaints': [{'complaint_id': 'CP0165',
    'complaint_type': 'AppIssue',
    'severity': 'Medium',
    'status': 'Open'}]},
 {'order_id': 'O00045',
  'customer': {'customer_id': 'C0326'},
  'complaints': [{'complaint_id': 'CP0163',
    'complaint_type': 'Billing',
    'severity': 'Medium',
    'status': 'Open'}]},
 {'order_id': 'O00050',
  'customer': {'customer_id': 'C0544'},
  'complaints': [{'complaint_id': 'CP0266',
    'complaint_type': 'Damage',
    'severity': 'High',
    'status': 'Open'}]},
 {'order_id': 'O00082',
  'customer': {'customer_id': 'C0531'},
  'complaints': [{'complaint_id': 'CP0225',
    'complaint_type': 'MissedPickup',
    'severity': 'Medium',
    'status': 'Open'}]},
 {'order_id': 'O00089',
  'customer': {'customer_id': 'C0339'},
  'complaints': [{'complaint_id': 'CP0294',
    'complaint_type': 'AppIssue',
    'severity': 'High',
    'status': 'Open'}]}]

In [ ]:
open_complaint_explain = collection.find(
    {"complaints.status": "Open"}
).explain()
open_complaint_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'complaints.status': {'$eq': 'Open'}},
  'indexFilterSet': False,
  'queryHash': '8A38CB5B',
  'planCacheShapeHash': '8A38CB5B',
  'planCacheKey': '4FDFCFA7',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'complaints.status': 1},
    'indexName': 'complaints.status_1',
    'isMultiKey': True,
    'multiKeyPaths': {'complaints.status': ['complaints']},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'complaints.status': ['["Open", "Open"]']}}},
  'rejectedPlans': []},
 'executionStats': {'executionSuccess': True,
  'nReturned': 54,
  'ex

In [ ]:
collection.distinct("app_events.event_type")

[None,
 'cancel_attempt',
 'chat_escalated',
 'chat_opened',
 'delivery_instruction_update',
 'eta_refresh',
 'payment_retry',
 'search_route',
 'track_order']

In [ ]:
# Optimised query: app events by event type
app_event_cases = list(collection.find(
    {"app_events.event_type": "payment_retry"},
    {
        "_id": 0,
        "order_id": 1,
        "customer.customer_id": 1,
        "app_events.event_id": 1,
        "app_events.event_type": 1,
        "app_events.api_latency_ms": 1,
        "app_events.success_flag": 1
    }
).limit(5))
app_event_cases

[{'order_id': 'O00011',
  'customer': {'customer_id': 'C0340'},
  'app_events': [{'event_id': 'AE00123',
    'event_type': 'payment_retry',
    'api_latency_ms': 1558,
    'success_flag': 1},
   {'event_id': 'AE00575',
    'event_type': 'track_order',
    'api_latency_ms': 456,
    'success_flag': 1}]},
 {'order_id': 'O00056',
  'customer': {'customer_id': 'C0271'},
  'app_events': [{'event_id': 'AE00369',
    'event_type': 'payment_retry',
    'api_latency_ms': 767,
    'success_flag': 1}]},
 {'order_id': 'O00078',
  'customer': {'customer_id': 'C0434'},
  'app_events': [{'event_id': 'AE00090',
    'event_type': 'payment_retry',
    'api_latency_ms': 820,
    'success_flag': 1}]},
 {'order_id': 'O00100',
  'customer': {'customer_id': 'C0300'},
  'app_events': [{'event_id': 'AE00562',
    'event_type': 'payment_retry',
    'api_latency_ms': 567,
    'success_flag': 0}]},
 {'order_id': 'O00114',
  'customer': {'customer_id': 'C0144'},
  'app_events': [{'event_id': 'AE00103',
    'event_

In [ ]:
app_event_explain = collection.find(
    {"app_events.event_type": "payment_retry"}
).explain()
app_event_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'app_events.event_type': {'$eq': 'payment_retry'}},
  'indexFilterSet': False,
  'queryHash': '7072FFEF',
  'planCacheShapeHash': '7072FFEF',
  'planCacheKey': '5ACE0E95',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'app_events.event_type': 1},
    'indexName': 'app_events.event_type_1',
    'isMultiKey': True,
    'multiKeyPaths': {'app_events.event_type': ['app_events']},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'app_events.event_type': ['["payment_retry", "payment_retry"]']}}},
  'rejectedPlans': []},
 'executionStats': {'ex

In [ ]:
# Optimised query: failed app events
failed_app_cases = list(collection.find(
    {"app_events.success_flag": 0},
    {
        "_id": 0,
        "order_id": 1,
        "customer.customer_id": 1,
        "app_events.event_id": 1,
        "app_events.event_type": 1,
        "app_events.success_flag": 1,
        "app_events.api_latency_ms": 1
    }
).limit(5))
failed_app_cases

[{'order_id': 'O00053',
  'customer': {'customer_id': 'C0054'},
  'app_events': [{'event_id': 'AE00582',
    'event_type': 'chat_escalated',
    'api_latency_ms': 1402,
    'success_flag': 0}]},
 {'order_id': 'O00100',
  'customer': {'customer_id': 'C0300'},
  'app_events': [{'event_id': 'AE00562',
    'event_type': 'payment_retry',
    'api_latency_ms': 567,
    'success_flag': 0}]},
 {'order_id': 'O00321',
  'customer': {'customer_id': 'C0016'},
  'app_events': [{'event_id': 'AE00492',
    'event_type': 'chat_escalated',
    'api_latency_ms': 732,
    'success_flag': 0}]},
 {'order_id': 'O00409',
  'customer': {'customer_id': 'C0065'},
  'app_events': [{'event_id': 'AE00277',
    'event_type': 'payment_retry',
    'api_latency_ms': 552,
    'success_flag': 0}]},
 {'order_id': 'O00414',
  'customer': {'customer_id': 'C0023'},
  'app_events': [{'event_id': 'AE00013',
    'event_type': 'chat_escalated',
    'api_latency_ms': 399,
    'success_flag': 0}]}]

In [ ]:
failed_app_explain = collection.find(
    {"app_events.success_flag": 0}
).explain()
failed_app_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'app_events.success_flag': {'$eq': 0}},
  'indexFilterSet': False,
  'queryHash': 'C68086FE',
  'planCacheShapeHash': 'C68086FE',
  'planCacheKey': '6802D33E',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'COLLSCAN',
   'filter': {'app_events.success_flag': {'$eq': 0}},
   'direction': 'forward'},
  'rejectedPlans': []},
 'executionStats': {'executionSuccess': True,
  'nReturned': 28,
  'executionTimeMillis': 1,
  'totalKeysExamined': 0,
  'totalDocsExamined': 1251,
  'executionStages': {'isCached': False,
   'stage': 'COLLSCAN',
   'filter': {'app_events.success_flag': {'$eq': 0}},
   'nReturned': 28,
   'executionTimeMillisEstimate': 0,
   'works': 1252,
   'advanced': 28,
   'needTime': 

In [ ]:
collection.distinct("pickup_zone")
collection.distinct("dropoff_zone")

['AIRPORT',
 'Airport',
 'CENTRAL',
 'Central',
 'Ctr',
 'EAST',
 'East',
 'NORTH',
 'North',
 'RiverSide',
 'Riverside',
 'SOUTH',
 'South',
 'WEST',
 'West',
 'north']

In [ ]:
# Optimised query: pickup and drop-off route search
route_zone_cases = list(collection.find(
    {
        "pickup_zone": "Central",
        "dropoff_zone": "North"
    },
    {
        "_id": 0,
        "order_id": 1,
        "service_type": 1,
        "pickup_zone": 1,
        "dropoff_zone": 1,
        "order_value": 1,
        "deliveries.delivery_status": 1
    }
).limit(10))
route_zone_cases

[{'order_id': 'O00448',
  'service_type': 'Medical',
  'pickup_zone': 'Central',
  'dropoff_zone': 'North',
  'order_value': 68.78,
  'deliveries': [{'delivery_status': 'OnTime'}]},
 {'order_id': 'O00486',
  'service_type': 'Retail',
  'pickup_zone': 'Central',
  'dropoff_zone': 'North',
  'order_value': 50.89,
  'deliveries': [{'delivery_status': 'OnTime'}]},
 {'order_id': 'O99999',
  'service_type': 'Business',
  'pickup_zone': 'Central',
  'dropoff_zone': 'North',
  'order_value': 75.5,
  'deliveries': [{'delivery_status': 'Failed'}]}]

In [ ]:
route_zone_explain = collection.find(
    {
        "pickup_zone": "Central",
        "dropoff_zone": "North"
    }
).explain()
route_zone_explain

{'explainVersion': '1',
 'queryPlanner': {'namespace': 'northstar_db.orders_operational_view',
  'parsedQuery': {'$and': [{'dropoff_zone': {'$eq': 'North'}},
    {'pickup_zone': {'$eq': 'Central'}}]},
  'indexFilterSet': False,
  'queryHash': '06131FCB',
  'planCacheShapeHash': '06131FCB',
  'planCacheKey': '58E97D1D',
  'optimizationTimeMillis': 0,
  'maxIndexedOrSolutionsReached': False,
  'maxIndexedAndSolutionsReached': False,
  'maxScansToExplodeReached': False,
  'prunedSimilarIndexes': False,
  'winningPlan': {'isCached': False,
   'stage': 'FETCH',
   'filter': {'pickup_zone': {'$eq': 'Central'}},
   'inputStage': {'stage': 'IXSCAN',
    'keyPattern': {'dropoff_zone': 1},
    'indexName': 'dropoff_zone_1',
    'isMultiKey': False,
    'multiKeyPaths': {'dropoff_zone': []},
    'isUnique': False,
    'isSparse': False,
    'isPartial': False,
    'indexVersion': 2,
    'direction': 'forward',
    'indexBounds': {'dropoff_zone': ['["North", "North"]']}}},
  'rejectedPlans': [{'is

In [ ]:
# List all indexes created on the collection

indexes = list(collection.list_indexes())
indexes

[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('order_id', 1)])), ('name', 'order_id_1')]),
 SON([('v', 2), ('key', SON([('customer.customer_id', 1)])), ('name', 'customer.customer_id_1')]),
 SON([('v', 2), ('key', SON([('service_type', 1)])), ('name', 'service_type_1')]),
 SON([('v', 2), ('key', SON([('pickup_zone', 1)])), ('name', 'pickup_zone_1')]),
 SON([('v', 2), ('key', SON([('dropoff_zone', 1)])), ('name', 'dropoff_zone_1')]),
 SON([('v', 2), ('key', SON([('deliveries.delivery_status', 1)])), ('name', 'deliveries.delivery_status_1')]),
 SON([('v', 2), ('key', SON([('deliveries.driver.driver_id', 1)])), ('name', 'deliveries.driver.driver_id_1')]),
 SON([('v', 2), ('key', SON([('deliveries.vehicle.vehicle_id', 1)])), ('name', 'deliveries.vehicle.vehicle_id_1')]),
 SON([('v', 2), ('key', SON([('deliveries.hub.hub_id', 1)])), ('name', 'deliveries.hub.hub_id_1')]),
 SON([('v', 2), ('key', SON([('complaints.severity', 1)])), ('name', 'com

In [ ]:
# Explain query using executionStats
explain_stats = db.command(
    "explain",
    {
        "find": "orders_operational_view",
        "filter": {"service_type": "Delivery"}
    },
    verbosity="executionStats"
)
explain_stats["executionStats"]

{'executionSuccess': True,
 'nReturned': 0,
 'executionTimeMillis': 0,
 'totalKeysExamined': 0,
 'totalDocsExamined': 0,
 'executionStages': {'isCached': False,
  'stage': 'FETCH',
  'nReturned': 0,
  'executionTimeMillisEstimate': 0,
  'works': 1,
  'advanced': 0,
  'needTime': 0,
  'needYield': 0,
  'saveState': 0,
  'restoreState': 0,
  'isEOF': 1,
  'docsExamined': 0,
  'alreadyHasObj': 0,
  'inputStage': {'stage': 'IXSCAN',
   'nReturned': 0,
   'executionTimeMillisEstimate': 0,
   'works': 1,
   'advanced': 0,
   'needTime': 0,
   'needYield': 0,
   'saveState': 0,
   'restoreState': 0,
   'isEOF': 1,
   'keyPattern': {'service_type': 1},
   'indexName': 'service_type_1',
   'isMultiKey': False,
   'multiKeyPaths': {'service_type': []},
   'isUnique': False,
   'isSparse': False,
   'isPartial': False,
   'indexVersion': 2,
   'direction': 'forward',
   'indexBounds': {'service_type': ['["Delivery", "Delivery"]']},
   'keysExamined': 0,
   'seeks': 1,
   'dupsTested': 0,
   'dups

In [ ]:
# Optimised query: find all orders linked to one customer

customer_search = list(collection.find(
    {"customer.customer_id": "C0001"},
    {
        "_id": 0,
        "order_id": 1,
        "service_type": 1,
        "pickup_zone": 1,
        "dropoff_zone": 1,
        "customer.customer_id": 1,
        "customer.customer_type": 1,
        "deliveries.delivery_status": 1,
        "complaints.status": 1
    }
).limit(10))

customer_search

[{'order_id': 'O00007',
  'service_type': 'Business',
  'pickup_zone': 'CENTRAL',
  'dropoff_zone': 'Airport',
  'customer': {'customer_id': 'C0001', 'customer_type': 'SME'},
  'deliveries': [{'delivery_status': 'Delayed'}],
  'complaints': [{'status': 'Resolved'}]},
 {'order_id': 'O00666',
  'service_type': 'Parcel',
  'pickup_zone': 'AIRPORT',
  'dropoff_zone': 'CENTRAL',
  'customer': {'customer_id': 'C0001', 'customer_type': 'SME'},
  'deliveries': [{'delivery_status': 'OnTime'}],
  'complaints': [{'status': 'Resolved'}]},
 {'order_id': 'O00721',
  'service_type': 'Retail',
  'pickup_zone': 'EAST',
  'dropoff_zone': 'Ctr',
  'customer': {'customer_id': 'C0001', 'customer_type': 'SME'},
  'deliveries': [],
  'complaints': []}]

In [ ]:
customer_explain = db.command(
    "explain",
    {
        "find": "orders_operational_view",
        "filter": {"customer.customer_id": "C001"}
    },
    verbosity="executionStats"
)

customer_explain["executionStats"]

{'executionSuccess': True,
 'nReturned': 0,
 'executionTimeMillis': 0,
 'totalKeysExamined': 0,
 'totalDocsExamined': 0,
 'executionStages': {'isCached': False,
  'stage': 'FETCH',
  'nReturned': 0,
  'executionTimeMillisEstimate': 0,
  'works': 1,
  'advanced': 0,
  'needTime': 0,
  'needYield': 0,
  'saveState': 0,
  'restoreState': 0,
  'isEOF': 1,
  'docsExamined': 0,
  'alreadyHasObj': 0,
  'inputStage': {'stage': 'IXSCAN',
   'nReturned': 0,
   'executionTimeMillisEstimate': 0,
   'works': 1,
   'advanced': 0,
   'needTime': 0,
   'needYield': 0,
   'saveState': 0,
   'restoreState': 0,
   'isEOF': 1,
   'keyPattern': {'customer.customer_id': 1},
   'indexName': 'customer.customer_id_1',
   'isMultiKey': False,
   'multiKeyPaths': {'customer.customer_id': []},
   'isUnique': False,
   'isSparse': False,
   'isPartial': False,
   'indexVersion': 2,
   'direction': 'forward',
   'indexBounds': {'customer.customer_id': ['["C001", "C001"]']},
   'keysExamined': 0,
   'seeks': 1,
   '